# Basic one site + bath


## Outline of the problem


We explore

- a fermonic system (single site) in contact with a fermionic bath
- the bath is formed by $n$ sites, discretising a custom spectral function $J(\omega)$
- the bath sites are all independent 
- the simplest approach is to simply **evolve** the correlation matrix

 $$C = \langle c_i^\dagger c_j\rangle$$

 where $c, c^\dagger $ are the annihilation/creation operators.

 Since the global Hamiltonian is in the form 

 $$ \hat{H} = \sum_{ij}^{n+1} H_{ij}  c_i^{\dagger}c_j $$

it is **quadratic** and by Wick's theorem *[find a good understandable reference]* this means that **higher-order correlations** can be expressed **solely** as **combinations** of the pair-wise correlation matrix $C$.

Note that the matrix $H_{ij}$ ($(n+1)\times (n+1)$) should be easy to express explicitely in terms of the couplings [do this below].

Given $\hat{H}$, we can then calculate the unitary $U$ as

 $$ U(t)= e^{-i \hat{H} t}$$


The evolution of an initial matrix $C_0= C(t=0)$ is then

$$C(t) = U C_0 U^{\dagger}$$


Which we we should be able to do explicitly in Julia or Python.


This notebook is the **Julia translation** of the original Python/NumPy implementation (the exploratory `pytorch` GPU benchmark from the Python notebook is not translated here, as requested).

We are now left with the problem of 

- specifying the Hamiltonian coefficients $H_{ij}$ (which are time independent)
- specifying the initial correlation matrix $C_0$

## Setting up the Hamiltonian

The Hamiltonian written above is determined by $H_{ij}$. This contains 

- the energy of the system $\epsilon$
- the system-bath couplings
- the individual bath site energies

So, the total hamiltonian should look like

$$\hat{H} = \epsilon \hat{s}^{\dagger}\hat{s} +\sum_k\sqrt{J_k}(\hat{b}_k^{\dagger} \hat{s}+ \hat{s}^{\dagger} \hat{b}_k)+\sum_k \omega_k \hat{b}_k^{\dagger} \hat{b}_k $$

where i am using $s$ for the site and $b$ for the bath. $\epsilon$, $J_k$ and $\omega$ are all real (Q: is this too restrictive?).

So let $i=0$ be the index for the system and $i=1,\dots, n$ be the index of for the bath (in Julia's 1-based indexing this becomes $i=1$ for the system and $i=2,\dots,n+1$ for the bath):

- H is then a symmetric matrix with zeros everywhere excpeting the diagional and the first row and column

To specify the couplings I need to choose a spectral function. I start with the semicircle of radius $D$ 

$$J(\omega) = \sqrt{1- (\omega/D)^2}$$


In [ ]:
import Pkg; 
Pkg.add("ColorSchemes")

In [ ]:
using LinearAlgebra
using Plots
using ColorSchemes

"Spectral function of choice"
J(omega; prefactor=0.005, D=1.0) = prefactor .* sqrt.(1.0 .- (omega ./ D) .^ 2)

n = 300 # an arbitrary choice for now, not too large
N = n + 1

H = zeros(ComplexF64, N, N)
epsilon = 0.0 # our scale of energies

D = 1.0 # choice of the bandwidth
omegas = collect(range(-D, D, length=n)) # discretisation of the bath
Js = J(omegas; D=D)

# first column (index 1 is the system, 2:end is the bath)
# scaled for correct thermodynamic limit, since the rates depend on j_i^2
H[2:end, 1] .= sqrt.(Js) ./ sqrt(n)
# first row
H[1, 2:end] .= sqrt.(Js) ./ sqrt(n)
# diagonal
H[diagind(H)] .= [epsilon; omegas]
println(H)

# plot the spectral function and its discretisation
_x = range(minimum(omegas), maximum(omegas), length=1000)
plot(_x, J(_x), label="J(ω)")
scatter!(omegas, Js, label="discretised")
xlabel!("ω")
ylabel!("J(ω)")

## Setting up the initial state


As we have a fermionic system, we assume that the initial state is governed by the **Fermi distribution**.


The assumption is that there are **no initial correlations**, so $C_0$ should be **diagonal** with only the **occupation numbers** on the diagonal.

These occupation numbers are determined by the Fermi-Diac statistics:

$$C_{ii}  = \dfrac{1}{1+e^{\beta E_i-\beta\mu}}$$

where $E_i$ are the energis of the sites (including the bath)

In [ ]:
C0 = zeros(ComplexF64, N, N)
# choosing the temperature
beta = 1.0
mu = 0.0
fermi(x) = 1 ./ (1 .+ exp.(beta .* x .- beta .* mu))

density_0 = 0
C0[diagind(C0)] .= [0.0; fermi.(omegas)]

C0

The matrix is initially diagonal in the **site basis**, not in the Hamiltonian basis.

In [ ]:
# Step 1: Diagonalize H
e_vals, U = eigen(Hermitian(H))  # H = U * Diagonal(e_vals) * U'

# Step 2: Transform C0 into the energy eigenbasis
C_energy_basis = U' * C0 * U

# Step 3: Check diagonality
off_diagonal_norm = norm(C_energy_basis - Diagonal(diag(C_energy_basis)))
println("Off-diagonal norm of C in energy basis: ", off_diagonal_norm)

In [ ]:
p1 = heatmap(log10.(abs.(C0) .+ 1e-6), yflip=true, color=:inferno, title="C in the original basis")
p2 = heatmap(log10.(abs.(C_energy_basis) .+ 1e-6), yflip=true, color=:inferno, title="C in the energy basis")
plot(p1, p2, layout=(1, 2), size=(1000, 450))

## Construct the unitary


The last missing bit is just the construction of the unitary.


The unitary can be constructed using the formula
$$U(t)= e^{-i \hat{H} t}$$

In Julia, `exp` applied to a `Matrix` computes the matrix exponential directly (equivalent to `scipy.linalg.expm`).

In [ ]:
unitary(t) = exp(-1im * H * t)

## Calculating observables

The time evolution is then trivial (but maybe inefficient).


In [ ]:
function evolve(C, t)
    U = unitary(t)
    return U * C * U'
end


We can calculate multiple correlation matrices at various times and try and plot how they vary over time.

I need to set up some tests to see if this makes sense.

In [ ]:
dt = 0.1
tmax = 1000.0
times = collect(0:dt:tmax-dt) # mirrors np.arange(0, tmax, dt)

In [ ]:
# direct calculation
# Cs = Dict(t => evolve(C0, t) for t in times)

In [ ]:
# supposedly faster iterative calculation
Udt = unitary(dt)
UdtT = Udt'
Css = Dict(0.0 => C0)
key = 0.0
for t in times[2:end]
    Css[t] = Udt * Css[key] * UdtT
    key = t
end

In [ ]:
# count = 0
# for (key, val) in Css
#     if count % 10 == 0
#         heatmap(log.(abs.(real.(val)) .+ 1e-12))
#         title!("t=$key")
#         savefig("C-time$key.png")
#     end
#     count += 1
# end

## Time evolution of the density on the system

In [ ]:
density = [Css[t][1, 1] for t in times]

In [ ]:
plot(times, real.(density), xlabel="Time", ylabel="Occupation on the system", legend=false)

The total system (site + finite bath) is closed and finite, so the dynamics seems (quasi)-periodic.

The system site cannot fully relax to a steady value like in an infinite bath (no true dissipation).

The oscillations shoudl go down as we increase the bath size.

## Overall occupation in the bath

In [ ]:
plt = plot(legend=false)
for t in times
    val = Css[t]
    c = get(ColorSchemes.coolwarm, t / tmax)
    plot!(plt, omegas, real.(diag(val[2:end, 2:end])), color=c)
end

u = range(-1.2, 1.2, length=50)
scatter!(plt, u, fermi.(u), color=:black, markersize=2, markerstrokewidth=0)
xlabel!("ω")
ylabel!("Occupation")
plt

## Preliminary things to understand

### On convergence

- How does the convergence depend on the quality of the time discretisation
- How does it truly depend on the number of bath sites (on CPU and GPUs)?
- How does it depend on alternative choices of the initial state
- How does it compare with alternative methods (e.g. the matrix product state (MPS) approach)? 

### On the physics

- What happens to the covariance matrix as we time evolve? Does it show any emergent structure? 
- How do these structures relate to the initial choice of the spectral function?
- Let's perturb the spectral function: can we see long-lived memory effects building up?
